## Deliverable 2

Imports 

In [19]:

import re
from pathlib import Path
import json
from typing import Any
import pandas as pd
import numpy as np


Task 1 -  File ingestion and regular expressions

In [ ]:

def read_file(filepath): #Reads a file and returns its content as a string
    path = Path(filepath)
    raw = path.read_bytes()

    try:
        text = raw.decode("utf-8") #Try to decode the bytes as UTF-8
        encoding = "utf-8"
    except UnicodeDecodeError:
        text = raw.decode("latin-1") #If UTF-8 decoding fails, decode the bytes as Latin-1
        encoding = "latin-1"

    return text, encoding



LOG_PATTERN = re.compile(
    r"\[(?P<timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\] " #Timestamp in the format YYYY-MM-DD HH:MM:SS
    r"\[(?P<register_id>\w+)\] " #Register ID
    r"item=(?P<item>.+?) "   #Item name 
    r"price=(?P<price>\d+(?:\.\d+)?) " #Price as a an int or decimal number
    r"(?P<currency>[A-Z]{3}) " #Curreny as a three uppercase letters
    r"\((?P<status>SUCCESS|REFUND|CANCELLED)\)" #Gives the status of the transaction 
)


def parse_log(text): #Defines a function that is goeing to return a list of dictionaries 
    records = []

    for line in text.splitlines(): #Removes whitespace
        match = LOG_PATTERN.fullmatch(line.strip())
        if match is None:
            continue  # støylinje, tom linje eller ugyldig linje

        record = match.groupdict()
        record["price"] = float(record["price"]) #Converts the price from a string to a float number
        records.append(record)

    return records


def parse_log_file(filepath): #Reds a log file and parses it in one step
    text, _encoding = read_file(filepath)
    return parse_log(text)


{'timestamp': '2026-10-15 08:30:00', 'register_id': 'POS_01', 'item': 'Latte', 'price': 4.5, 'currency': 'EUR', 'status': 'SUCCESS'}
{'timestamp': '2026-10-15 08:31:12', 'register_id': 'POS_02', 'item': 'Iced Latte', 'price': 55.0, 'currency': 'NOK', 'status': 'REFUND'}


Task 2 - Recursive utility functions

In [ ]:

def find_files(directory: str | Path) -> list[Path]:

    directory = Path(directory) #Converts to a path object 
    log_files: list[Path] = []

    for item in directory.iterdir(): #Iterates through the items in the directory
        if item.is_dir():
            log_files.extend(find_files(item))
        elif item.is_file() and item.suffix == ".log":
            log_files.append(item)

    return log_files


def find_active_registers(data: Any) -> set[str]:
    active_ids: set[str] = set()

    if isinstance(data, dict): #Checks if the data is a dictionary
        if data.get("active") is True and "id" in data: #Checks if the register is active and has an ID
            active_ids.add(data["id"])

        for value in data.values(): #Serches for active registers in the dictonary 
            active_ids |= find_active_registers(value)

    elif isinstance(data, list):
        for item in data: #Search each item in the list 
            active_ids |= find_active_registers(item)

    return active_ids #Returns a set of active register IDs


def load_active_registers(
    config_path: str | Path = "data/store_config.json",
) -> set[str]:
    
    path = Path(config_path) #Converts to a path object

    with path.open("r", encoding="utf-8") as file: #Opens the file in read mode with UTF-8 encoding
        config = json.load(file)

    return find_active_registers(config) #Returns all active register IDs 




Task 3 - Functions for cleaning and analysing data using NumPy

In [22]:

def remove_price_outliers(prices: np.ndarray) -> np.ndarray:
    
    """
    Removes price outliers by using the 3-sigma rule
    Prices that are more then 3 standard derivation from the mean are replaced with the median price
    """
    
    prices = np.asarray(prices, dtype = float)
    
    mean = np.mean(prices)
    std = np.std(prices)
    
    #Masks that find outliners
    outliers_mask = np.abs(prices - mean) > 3 * std
    #Finds the valid prices
    valid_mask = ~outliers_mask 
    
    #Finds the median
    valid_prices = prices[valid_mask]
    median = np.median(valid_prices)
    
    #Replace abberation with median by the valid prices 
    cleaned_prices = prices.copy()
    cleaned_prices[outliers_mask] = median
    
    return cleaned_prices

Task 4 - Data analysis using Pandas

In [23]:
def calculate_refund_ratio(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calculate refund ratios for all items.
    Counts succesfull transactions and refunded transactions for each item
    """
    
    #Find numbers sucessfully and refund per hour
    sales = (df[df["status"] == "SUCCESS"]
             .groupby("item")
             .size()
             )
    
    refund = (df[df["status"] == "REFUND"]
              .groupby("item")
              .size()
              )

    #Link togheter as an doubble DataFrame
    counts = pd.DataFrame({
        "sales": sales,
        "refund": refund
        }).fillna(0)
    
    #Calculate ratio
    counts["refund_ratio"] = compute_refund_ratio(
        counts["sales"].to_numpy(),
        counts["refund"].to_numpy()
        )
    
    return counts

Task 5 - Create full workflow

In [1]:
def main() -> None:
    """
    Execute the complete data-processing workflow
    """
    
    log_files = find_files("data/2026")
    transactions = parse_log(log_files)
    df = create_dataframe(transactions)
    
    #Remove price-outliers
    df["price"] = remove_price_outliers(df["price"].to_numpy())
    
    #Rectify valuta 
    df.loc[df["currency"] == "NOK", "price"] *= 10.0
    
    
    #Rapports
    hourly = hourly_revenue(df)
    pivot = create_pivot_table(df)
    refunds = calculate_refund_ratio(df)
    
    #Save files
    df.to_csv("cleaned_transactions.csv", index = False)
    hourly.to_json("hourly_revenue.json")
    pivot.to_csv("status_pivot_table.csv")
    refunds.to_csv("item_refund_ratios.csv")